In [1]:
import os
import psycopg2
import pandas as pd
import numpy as np
import joblib
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

load_dotenv(dotenv_path='../../.env') 

print("1. Connecting securely to PostgreSQL Data Warehouse...")
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD", ""), 
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)

query = """
    SELECT 
        term_months, 
        disbursement_gross, 
        sba_appv_amount, 
        rev_line_cr_flag, 
        loan_status 
    FROM fact_credit_portfolio
    WHERE loan_status IN ('P I F', 'CHGOFF', 'PIF');
"""

cursor = conn.cursor()
cursor.execute(query)
columns = [desc[0] for desc in cursor.description]
df = pd.DataFrame(cursor.fetchall(), columns=columns)
conn.close()

print(f" -> Extracted {len(df):,} pristine records.")

1. Connecting securely to PostgreSQL Data Warehouse...
 -> Extracted 455,689 pristine records.


In [2]:
print("2. Feature Engineering...")
# Create a binary target variable (1 = Default/Charge-Off, 0 = Paid In Full)
df['target'] = df['loan_status'].apply(lambda x: 1 if 'CHGOFF' in x else 0)

# Feature 1: Bank Risk Exposure (Percentage of loan guaranteed by the government)
df['sba_guarantee_ratio'] = df['sba_appv_amount'] / df['disbursement_gross']

# Feature 2: Clean Categorical Encoding ('Y'/'N' to 1/0)
df['is_revolving_line'] = df['rev_line_cr_flag'].apply(lambda x: 1 if x == 'Y' else 0)

# Select final model features
features = ['term_months', 'disbursement_gross', 'sba_guarantee_ratio', 'is_revolving_line']
X = df[features]
y = df['target']

2. Feature Engineering...


In [3]:
print("3. Training the Logistic Regression Model...")
# Split into 80% training data, 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the financial data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model with balanced class weights
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
print(" -> Model training complete.")

3. Training the Logistic Regression Model...
 -> Model training complete.


In [4]:
print("4. Model Evaluation & Business Metrics...")
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"\n--- Final ROC-AUC Score: {roc_auc:.4f} ---")

4. Model Evaluation & Business Metrics...
--- Confusion Matrix ---
[[50501 20740]
 [ 4180 15717]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.71      0.80     71241
           1       0.43      0.79      0.56     19897

    accuracy                           0.73     91138
   macro avg       0.68      0.75      0.68     91138
weighted avg       0.82      0.73      0.75     91138


--- Final ROC-AUC Score: 0.8193 ---


In [5]:
print("5. Serializing and Exporting Model for Production...")
# Save the model and the scaler to our isolated saved_models directory
joblib.dump(model, '../saved_models/logistic_credit_scorer.joblib')
joblib.dump(scaler, '../saved_models/financial_scaler.joblib')
print(" -> Model successfully saved to models/saved_models/")

5. Serializing and Exporting Model for Production...
 -> Model successfully saved to models/saved_models/
